# LangGraph Streaming: State Snapshots and Node Updates

| Field | Value |
|---|---|
| Stage | LangGraph and agentic foundations |
| Difficulty | Intermediate |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Streaming is an event contract. Consumers must know whether each event is a full state snapshot, a node update, or a model token.

## 30-Second Summary

This notebook builds a two-node offline LangGraph and compares `stream_mode='values'` with `stream_mode='updates'`. It reconstructs final state from updates and verifies parity with normal invocation.

## Why This Matters

Streaming improves perceived latency and observability, but confusing deltas with full state causes missing fields, duplicate rendering, and fragile clients.

## Scope

| Covers | Does not cover |
|---|---|
| Typed state, deterministic nodes, values/updates modes, reconstruction, cancellation notes | Hosted token stream, UI framework, distributed backpressure |


## Mental Model

```text
input state -> prepare node -> answer node -> final state
                | updates       | updates
                `------ values snapshots ------`
```


In [1]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class State(TypedDict, total=False):
    text: str
    normalized: str
    answer: str

def prepare(state: State) -> State:
    return {"normalized": state["text"].strip().lower()}

def answer(state: State) -> State:
    return {"answer": f"Echo: {state['normalized']}"}


## How It Works

`values` emits the accumulated state after each transition, including the initial input. `updates` emits only each node's returned delta, keyed by node name. Token streaming is a separate provider/model capability and should not be conflated with state streaming.


## Baseline

Normal invocation returns only final state. It is the correctness reference but exposes no intermediate progress.


In [2]:
builder = StateGraph(State)
builder.add_node("prepare", prepare)
builder.add_node("answer", answer)
builder.add_edge(START, "prepare")
builder.add_edge("prepare", "answer")
builder.add_edge("answer", END)
graph = builder.compile()

input_state = {"text": "  Hello Streaming  "}
final_state = graph.invoke(input_state)
final_state


{'text': '  Hello Streaming  ',
 'normalized': 'hello streaming',
 'answer': 'Echo: hello streaming'}

## Technique Implementation

We collect both stream modes from the same input. The notebook keeps event lists short and inspects their shape rather than printing long model traces.


In [3]:
value_events = list(graph.stream(input_state, stream_mode="values"))
update_events = list(graph.stream(input_state, stream_mode="updates"))
{"values": value_events, "updates": update_events}


{'values': [{'text': '  Hello Streaming  '},
  {'text': '  Hello Streaming  ', 'normalized': 'hello streaming'},
  {'text': '  Hello Streaming  ',
   'normalized': 'hello streaming',
   'answer': 'Echo: hello streaming'}],
 'updates': [{'prepare': {'normalized': 'hello streaming'}},
  {'answer': {'answer': 'Echo: hello streaming'}}]}

## Controlled Experiment

A client reconstructs state by starting with the input and merging each node update. The reconstructed state must equal both the final `values` event and ordinary invocation.


In [4]:
reconstructed = dict(input_state)
for event in update_events:
    for node_update in event.values():
        reconstructed.update(node_update)

results = {
    "value_event_count": len(value_events),
    "update_event_count": len(update_events),
    "final_value": value_events[-1],
    "reconstructed": reconstructed,
    "invoke_result": final_state,
}
results


{'value_event_count': 3,
 'update_event_count': 2,
 'final_value': {'text': '  Hello Streaming  ',
  'normalized': 'hello streaming',
  'answer': 'Echo: hello streaming'},
 'reconstructed': {'text': '  Hello Streaming  ',
  'normalized': 'hello streaming',
  'answer': 'Echo: hello streaming'},
 'invoke_result': {'text': '  Hello Streaming  ',
  'normalized': 'hello streaming',
  'answer': 'Echo: hello streaming'}}

## Evaluation

The graph emits **3 value snapshots** (input plus two nodes) and **2 update events**. Merging updates reconstructs exactly the same final state as `invoke`. This proves state-event semantics for the local graph, not network token behavior.


In [5]:
assert results["value_event_count"] == 3
assert results["update_event_count"] == 2
assert results["reconstructed"] == results["final_value"] == results["invoke_result"]
assert [next(iter(event)) for event in update_events] == ["prepare", "answer"]
print("LangGraph streaming checks passed.")


LangGraph streaming checks passed.


## Decision Guide

| Consumer need | Mode |
|---|---|
| Render current complete workflow state | `values` |
| Observe node-by-node deltas | `updates` |
| Show model text as generated | Token/message streaming |
| Durable audit | Persist selected structured events |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Fields disappear | Update treated as full state | Merge by documented reducer |
| UI duplicates content | Snapshots appended as deltas | Replace vs append by event type |
| Work continues after disconnect | Cancellation not propagated | Cooperative cancellation/timeouts |
| Producer overwhelms client | No backpressure/bounds | Buffer limits and coalescing |


## Production Notes

### Observability
Include run/thread ID, node, sequence, event type, timestamps, and terminal status.

### Safety and Guardrails
Do not stream hidden prompts, secrets, or unrestricted tool payloads.

### Latency and Cost
Streaming changes delivery, not model compute cost; measure time-to-first-event and total duration separately.


## Practice

Add a third node that records string length, then update the expected event counts and reconstruction check.

## Recall

Toggle - Recall: What does `updates` emit?
Each node's state delta keyed by node name.

Toggle - Recall: Does streaming reduce total compute?
Not necessarily; it mainly improves progressive delivery and observability.

## Sources

- [LangGraph streaming documentation](https://docs.langchain.com/oss/python/langgraph/streaming)
- [LangGraph graph API](https://reference.langchain.com/python/langgraph/graphs/)

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the installed LangGraph state modes | Add token streaming, cancellation, and backpressure fixtures |
